# Combined IDRiD + APTOS Federated Learning Sweep

This notebook contains the complete federated learning sweep on the combined IDRiD + APTOS dataset.

### Sweep Specifications:
- **Grid**: Dirichlet $\alpha \in \{0.1, 0.3, 1.0\}$, Proximal Regularization $\mu \in \{0.0, 0.1, 1.0\}$, Seeds $\in \{42, 123, 777\}$.
- **FL Strategy**: `fedavg` (for $\mu=0.0$) and `fedprox` (for $\mu > 0.0$).
- **Clients**: 3 clients.
- **Global Evaluator**: Shared IDRiD test dataset (103 images).
- **Hyperparameters**: Pretrained EfficientNet-B0, AdamW optimizer ($lr=1e-4$, $weight\_decay=1e-2$), CosineAnnealingLR ($T_{max}=25$), batch size 8, 10 rounds, 3 local epochs.
- **Dataloading**: Parallel dataloading (`num_workers=4`, `pin_memory=True`), dynamic resizing to 224x224.
- **Logging**: Saves progress summaries and round metrics to `results/combined_experiment_summary.csv` and `results/combined_round_metrics.csv`.
- **Checkpointing**: Checkpoint after each run to `checkpoints/completed.json` for pause/resume.

In [1]:
import sys, os, time, json, random

# Force Hugging Face Hub to run in offline mode to load weights from the local cache
os.environ['HF_HUB_OFFLINE'] = '1'
print('[INFO] Set HF_HUB_OFFLINE = 1 to load model weights from the local cache.')

# Fix httpx ValueError with socks:// proxy schemes by converting them to socks5://
for key in ['http_proxy', 'https_proxy', 'all_proxy', 'HTTP_PROXY', 'HTTPS_PROXY', 'ALL_PROXY']:
    if key in os.environ and os.environ[key].startswith('socks://'):
        os.environ[key] = os.environ[key].replace('socks://', 'socks5://')
        print(f'[INFO] Converted proxy {key} from socks:// to socks5:// to support httpx.')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

NOTEBOOK_DIR = os.getcwd()
DATA_DIR = os.path.join(NOTEBOOK_DIR, 'data')
RESULTS_DIR = os.path.join(NOTEBOOK_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

[INFO] Set HF_HUB_OFFLINE = 1 to load model weights from the local cache.
[INFO] Converted proxy all_proxy from socks:// to socks5:// to support httpx.
[INFO] Converted proxy ALL_PROXY from socks:// to socks5:// to support httpx.
Python: 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
PyTorch: 2.5.1+cu121
Device: cuda
GPU: NVIDIA RTX A2000 12GB


In [2]:
train_transform = transforms.Compose([
    transforms.Resize((240,240)),
    transforms.RandomCrop((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

class CombinedPartitionDataset(Dataset):
    def __init__(self, json_path, transform=None):
        with open(json_path, 'r') as f:
            self.data = json.load(f)
        self.transform = transform
    def __len__(self): 
        return len(self.data)
    def __getitem__(self, idx):
        record = self.data[idx]
        img_path = os.path.join(NOTEBOOK_DIR, record['relative_path'])
        img = Image.open(img_path).convert('RGB')
        if self.transform: 
            img = self.transform(img)
        return img, int(record['grade'])

class CombinedTestDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        df = pd.read_csv(csv_path)[['Image name', 'Retinopathy grade']].dropna()
        self.image_names = df['Image name'].values
        self.labels = df['Retinopathy grade'].values.astype(int)
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): 
        return len(self.image_names)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.image_names[idx]+'.jpg')
        img = Image.open(img_path).convert('RGB')
        if self.transform: 
            img = self.transform(img)
        return img, int(self.labels[idx])

In [3]:
def get_weights(model):
    return [v.cpu().numpy() for v in model.state_dict().values()]

def set_weights(model, weights):
    sd = dict(zip(model.state_dict().keys(), [torch.tensor(w) for w in weights]))
    model.load_state_dict(sd, strict=True)

def train_fedprox(model, dataloader, global_weights, mu, lr, epochs, device, client_id=None):
    gsd = dict(zip(model.state_dict().keys(), global_weights))
    gparams = [torch.tensor(gsd[n]).to(device) for n, _ in model.named_parameters()]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=25)
    crit = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(epochs):
        if client_id is not None:
            print(f'      - Client {client_id} training epoch {epoch+1}/{epochs} ({len(dataloader)} batches)...')
            sys.stdout.flush()
        for bx, by in dataloader:
            bx, by = bx.to(device), by.to(device).long()
            opt.zero_grad()
            loss = crit(model(bx), by)
            if mu > 0:
                prox = sum(torch.norm(lp - gp)**2 for lp, gp in zip(model.parameters(), gparams))
                loss = loss + (mu/2)*prox
            loss.backward(); opt.step()
        scheduler.step()
    return get_weights(model), len(dataloader.dataset)

def evaluate_model(model, dataloader, device):
    model.eval(); total_loss=0.0; correct=0; total=0
    crit = nn.CrossEntropyLoss()
    with torch.no_grad():
        for bx, by in dataloader:
            bx, by = bx.to(device), by.to(device).long()
            out = model(bx)
            total_loss += crit(out, by).item() * len(by)
            correct += (out.argmax(1)==by).sum().item(); total += len(by)
    return (correct/total, total_loss/total) if total>0 else (0.0, 0.0)

def compute_l2_drift(lw, gw):
    return float(np.sqrt(sum(np.sum((a-b)**2) for a,b in zip(lw,gw))))

COMPLETED_FILE = os.path.join(NOTEBOOK_DIR, 'checkpoints', 'completed.json')
os.makedirs(os.path.dirname(COMPLETED_FILE), exist_ok=True)

def is_completed(eid):
    if not os.path.exists(COMPLETED_FILE): return False
    try:
        with open(COMPLETED_FILE) as f: return eid in json.load(f)
    except: return False

def mark_completed(eid):
    done = []
    if os.path.exists(COMPLETED_FILE):
        try:
            with open(COMPLETED_FILE) as f: done = json.load(f)
        except: pass
    done.append(eid)
    tmp = COMPLETED_FILE+'.tmp'
    with open(tmp,'w') as f: json.dump(done, f, indent=1)
    os.replace(tmp, COMPLETED_FILE)

In [4]:
print('[INFO] Loading shared test dataset...')
test_csv_path = os.path.join(DATA_DIR, 'idrid_raw', 'B. Disease Grading', '2. Groundtruths', 'b. IDRiD_Disease Grading_Testing Labels.csv')
test_img_dir = os.path.join(DATA_DIR, 'idrid_raw', 'B. Disease Grading', '1. Original Images', 'b. Testing Set')
test_dataset = CombinedTestDataset(test_csv_path, test_img_dir, transform=eval_transform)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
print(f'[OK] Test dataset loaded: {len(test_dataset)} images.')

[INFO] Loading shared test dataset...
[OK] Test dataset loaded: 103 images.


In [ ]:
import os, sys
os.environ['HF_HUB_OFFLINE'] = '1'

SWEEP_CONFIG = {
    'num_clients': 3, 'num_rounds': 10,
    'local_epochs': 3, 'batch_size': 8, 'lr': 1e-4
}

all_configs = []
for alpha in [0.1, 0.3, 1.0]:
    for mu in [0.0, 0.1, 1.0]:
        for seed in [42, 123, 777]:
            strategy = 'fedavg' if mu == 0.0 else 'fedprox'
            all_configs.append({
                'alpha': alpha, 'mu': mu, 'seed': seed, 'strategy': strategy
            })

total_rounds = len(all_configs) * SWEEP_CONFIG['num_rounds']
completed_rounds = 0
round_times = []
runtimes = []

def save_experiment_results(result, base_dir='.'):
    rows = []
    for rm in result['round_metrics']:
        cm = rm['client_metrics']
        rows.append({
            'experiment_id': result['experiment_id'],
            'round': rm['round'],
            'global_accuracy': rm['global_accuracy'],
            'global_loss': rm['global_loss'],
            'mean_train_loss': rm['mean_train_loss'],
            'mean_drift': rm['mean_drift'],
            'max_drift': rm['max_drift'],
            'client_0_local_on_test_accuracy': cm[0]['local_test_acc'],
            'client_0_local_on_test_loss':     cm[0]['local_test_loss'],
            'client_1_local_on_test_accuracy': cm[1]['local_test_acc'],
            'client_1_local_on_test_loss':     cm[1]['local_test_loss'],
            'client_2_local_on_test_accuracy': cm[2]['local_test_acc'],
            'client_2_local_on_test_loss':     cm[2]['local_test_loss'],
            'client_0_drift_diagnostic': cm[0]['drift_diagnostic_acc'],
            'client_1_drift_diagnostic': cm[1]['drift_diagnostic_acc'],
            'client_2_drift_diagnostic': cm[2]['drift_diagnostic_acc'],
            'alpha': result['alpha'], 'mu': result['mu'],
            'seed': result['seed'], 'strategy': result['strategy'],
            'straggler_fraction': 0.0
        })
    rpath = os.path.join(base_dir, 'results', 'combined_round_metrics.csv')
    rdf = pd.DataFrame(rows)
    rdf.to_csv(rpath, mode='a', header=not os.path.exists(rpath), index=False)

    spath = os.path.join(base_dir, 'results', 'combined_experiment_summary.csv')
    srow = {
        'experiment_id': result['experiment_id'],
        'alpha': result['alpha'], 'mu': result['mu'],
        'seed': result['seed'], 'strategy': result['strategy'],
        'straggler_fraction': 0.0,
        'final_global_accuracy': result['final_accuracy'],
        'best_global_accuracy': result['best_accuracy'],
        'convergence_round': result['convergence_round'],
        'local_on_test_accuracy_std': result['accuracy_std'],
        'worst_client_local_on_test_accuracy': result['worst_client_accuracy'],
        'per_client_local_on_test': str(result['per_client_local_on_test']),
        'runtime_seconds': result['runtime_seconds']
    }
    sdf = pd.DataFrame([srow])
    sdf.to_csv(spath, mode='a', header=not os.path.exists(spath), index=False)

print(f'[INFO] Starting combined dataset sweep execution over {len(all_configs)} configurations...')
sys.stdout.flush()

for exp_idx, conf in enumerate(all_configs):
    alpha, mu, seed, strategy = conf['alpha'], conf['mu'], conf['seed'], conf['strategy']
    exp_id = f'alpha{alpha}_mu{mu}_seed{seed}_{strategy}_straggler0.0_combined'
    
    if is_completed(exp_id):
        completed_rounds += SWEEP_CONFIG['num_rounds']
        print(f'[SKIP] {exp_id}')
        sys.stdout.flush()
        continue
        
    print(f'\n--- Starting Experiment {exp_idx+1}/{len(all_configs)}: alpha={alpha} mu={mu} seed={seed} strategy={strategy} ---')
    sys.stdout.flush()
    
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed(seed)
    
    client_loaders = {}
    eval_train_loaders = {}
    for cid in range(SWEEP_CONFIG['num_clients']):
        mpath = os.path.join(DATA_DIR, 'combined_partitions', f'alpha_{alpha}', f'client_{cid}.json')
        ds_train = CombinedPartitionDataset(mpath, transform=train_transform)
        client_loaders[cid] = DataLoader(ds_train, batch_size=SWEEP_CONFIG['batch_size'], shuffle=True, num_workers=4, pin_memory=True)
        
        ds_eval = CombinedPartitionDataset(mpath, transform=eval_transform)
        eval_train_loaders[cid] = DataLoader(ds_eval, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
        
    global_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5).to(device)
    global_weights = get_weights(global_model)
    round_metrics_list = []
    
    t_exp_start = time.time()
    for rnd in range(1, SWEEP_CONFIG['num_rounds'] + 1):
        r_start = time.time()
        client_updates = []
        round_client_metrics = []
        
        for cid in range(SWEEP_CONFIG['num_clients']):
            loader = client_loaders[cid]
            eval_loader = eval_train_loaders[cid]
            
            local_model = timm.create_model('efficientnet_b0', num_classes=5).to(device)
            set_weights(local_model, global_weights)
            
            local_weights, n_samples = train_fedprox(
                local_model, loader, global_weights,
                mu=mu, lr=SWEEP_CONFIG['lr'], epochs=SWEEP_CONFIG['local_epochs'], device=device, client_id=cid
            )
            
            local_test_acc, local_test_loss = evaluate_model(local_model, test_loader, device)
            _, train_loss = evaluate_model(local_model, eval_loader, device)
            drift = compute_l2_drift(local_weights, global_weights)
            
            client_updates.append((local_weights, n_samples))
            round_client_metrics.append({
                'local_test_acc': local_test_acc,
                'local_test_loss': local_test_loss,
                'drift_diagnostic_acc': 0.0,
                'drift_diagnostic_loss': 0.0,
                'drift': drift,
                'train_loss': train_loss
            })
            del local_model
            
        # Aggregation
        global_weights = (lambda updates: [
            sum(w[i]*(n/sum(nn for _,nn in updates)) for w,n in updates)
            for i in range(len(updates[0][0]))
        ])(client_updates)
        set_weights(global_model, global_weights)
        
        global_acc, global_loss = evaluate_model(global_model, test_loader, device)
        
        for cid in range(SWEEP_CONFIG['num_clients']):
            eval_loader = eval_train_loaders[cid]
            d_acc, d_loss = evaluate_model(global_model, eval_loader, device)
            round_client_metrics[cid]['drift_diagnostic_acc'] = d_acc
            round_client_metrics[cid]['drift_diagnostic_loss'] = d_loss
            
        mean_drift = float(np.mean([m['drift'] for m in round_client_metrics]))
        max_drift  = float(np.max( [m['drift'] for m in round_client_metrics]))
        mean_train = float(np.mean([m['train_loss'] for m in round_client_metrics]))
        
        round_metrics_list.append({
            'round': rnd,
            'global_accuracy': global_acc,
            'global_loss': global_loss,
            'mean_train_loss': mean_train,
            'mean_drift': mean_drift,
            'max_drift': max_drift,
            'client_metrics': round_client_metrics
        })
        
        r_elapsed = time.time() - r_start
        round_times.append(r_elapsed)
        completed_rounds += 1
        
        remaining_rounds = total_rounds - completed_rounds
        # Use the average of up to the last 15 rounds to compute ETA
        mean_round_time = np.mean(round_times[-15:])
        eta_hours = (remaining_rounds * mean_round_time) / 3600.0
        
        local_accs = [m['local_test_acc'] for m in round_client_metrics]
        worst_client_acc = np.min(local_accs)
        
        print(f'[Exp {exp_idx+1}/{len(all_configs)} | Round {rnd}/{SWEEP_CONFIG["num_rounds"]} | alpha={alpha} mu={mu} seed={seed}]')
        print(f'Global: {global_acc:.4f} | Worst client: {worst_client_acc:.4f} | ETA: ~{eta_hours:.1f}hrs')
        sys.stdout.flush()
        
    elapsed_exp = time.time() - t_exp_start
    final_local_accs = [m['local_test_acc'] for m in round_metrics_list[-1]['client_metrics']]
    
    res = {
        'alpha': alpha, 'mu': mu, 'seed': seed, 'strategy': strategy,
        'final_accuracy': round_metrics_list[-1]['global_accuracy'],
        'best_accuracy': max(rm['global_accuracy'] for rm in round_metrics_list),
        'convergence_round': next((rm['round'] for rm in round_metrics_list if rm['global_accuracy'] >= 0.95*round_metrics_list[-1]['global_accuracy']), None),
        'accuracy_std': float(np.std(final_local_accs)),
        'worst_client_accuracy': float(np.min(final_local_accs)),
        'per_client_local_on_test': final_local_accs,
        'round_metrics': round_metrics_list,
        'experiment_id': exp_id,
        'runtime_seconds': elapsed_exp
    }
    
    save_experiment_results(res, NOTEBOOK_DIR)
    mark_completed(exp_id)
    del global_model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

[INFO] Starting combined dataset sweep execution over 27 configurations...

--- Starting Experiment 1/27: alpha=0.1 mu=0.0 seed=42 strategy=fedavg ---
      - Client 0 training epoch 1/3 (189 batches)...
      - Client 0 training epoch 2/3 (189 batches)...
      - Client 0 training epoch 3/3 (189 batches)...
      - Client 1 training epoch 1/3 (210 batches)...
      - Client 1 training epoch 2/3 (210 batches)...
      - Client 1 training epoch 3/3 (210 batches)...
      - Client 2 training epoch 1/3 (112 batches)...
      - Client 2 training epoch 2/3 (112 batches)...
      - Client 2 training epoch 3/3 (112 batches)...
[Exp 1/27 | Round 1/10 | alpha=0.1 mu=0.0 seed=42]
Global: 0.3495 | Worst client: 0.3495 | ETA: ~28.3hrs
      - Client 0 training epoch 1/3 (189 batches)...
      - Client 0 training epoch 2/3 (189 batches)...
      - Client 0 training epoch 3/3 (189 batches)...
      - Client 1 training epoch 1/3 (210 batches)...
      - Client 1 training epoch 2/3 (210 batches)...
  

In [ ]:
summary_path = os.path.join(NOTEBOOK_DIR, 'results', 'combined_experiment_summary.csv')
if os.path.exists(summary_path):
    df = pd.read_csv(summary_path)
    print('=== COMBINED EXPERIMENT SUMMARY TABLE ===')
    # Print or display the pandas table
    from IPython.display import display
    display(df)
else:
    print('[INFO] No combined_experiment_summary.csv found yet. Complete the sweep to generate the summary table.')